In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [2]:
batch_size = 2
seq_length = 5
embedding_dim = 8

# Random input embeddings
x = torch.randn(batch_size, seq_length, embedding_dim)

print("Input Shape:", x.shape)
print(x)

Input Shape: torch.Size([2, 5, 8])
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431,
          -1.6047],
         [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,
           0.7624],
         [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,
           1.6806],
         [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,
           0.8599],
         [-1.3847, -0.8712, -0.2234,  1.7174,  0.3189, -0.4245,  0.3057,
          -0.7746]],

        [[-1.5576,  0.9956, -0.8798, -0.6011, -1.2742,  2.1228, -1.2347,
          -0.4879],
         [-0.9138, -0.6581,  0.0780,  0.5258, -0.4880,  1.1914, -0.8140,
          -0.7360],
         [-1.4032,  0.0360, -0.0635,  0.6756, -0.0978,  1.8446, -1.1845,
           1.3835],
         [ 1.4451,  0.8564,  2.2181,  0.5232,  0.3466, -0.1973, -1.0546,
           1.2780],
         [-0.1722,  0.5238,  0.0566,  0.4263,  0.5750, -0.6417, -2.2064,
          -0.7508]]])


In [3]:
class SelfAttention(nn.Module):

    def __init__(self, embedding_dim):
        super(SelfAttention, self).__init__()

        self.query = nn.Linear(embedding_dim, embedding_dim)
        self.key = nn.Linear(embedding_dim, embedding_dim)
        self.value = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Dot Product
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # Scaling
        scores = scores / (K.size(-1) ** 0.5)

        # Softmax
        attention_weights = F.softmax(scores, dim=-1)

        # Context Vector
        output = torch.matmul(attention_weights, V)

        return output, attention_weights

In [4]:
self_attention = SelfAttention(embedding_dim)

output, weights = self_attention(x)

print("Output Shape:", output.shape)
print(output)

print("\nAttention Weights Shape:", weights.shape)
print(weights)

Output Shape: torch.Size([2, 5, 8])
tensor([[[ 0.3196,  0.5913,  0.1112,  0.2385, -0.2364,  0.2817,  0.7342,
          -0.3859],
         [ 0.3891,  0.2901,  0.0545, -0.0274, -0.3973,  0.0152,  0.5125,
          -0.2799],
         [ 0.5068,  0.1609, -0.1646, -0.0736, -0.1448, -0.1097,  0.1327,
          -0.3663],
         [ 0.4821,  0.0985, -0.1619, -0.1848, -0.1952, -0.1710,  0.1417,
          -0.3479],
         [ 0.4797,  0.2599,  0.0267, -0.0697, -0.4437, -0.0357,  0.4457,
          -0.2539]],

        [[ 0.8641,  0.2788, -0.0943, -0.1151, -0.9724, -0.7465, -0.0080,
           0.1187],
         [ 0.8726,  0.2821, -0.0911, -0.1670, -0.9276, -0.7541,  0.0245,
           0.1118],
         [ 0.8129,  0.1206, -0.1593, -0.1441, -1.0795, -0.8394, -0.0497,
           0.1250],
         [ 0.7827,  0.1877, -0.1380, -0.0693, -1.0269, -0.7775, -0.0561,
           0.1096],
         [ 0.9131,  0.3345, -0.0662, -0.2039, -0.8769, -0.7365,  0.0599,
           0.1121]]], grad_fn=<UnsafeViewBackward0>)

In [5]:
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super(MultiHeadSelfAttention, self).__init__()

        assert embedding_dim % num_heads == 0

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.query = nn.Linear(embedding_dim, embedding_dim)
        self.key = nn.Linear(embedding_dim, embedding_dim)
        self.value = nn.Linear(embedding_dim, embedding_dim)

        self.fc_out = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):

        batch_size = x.shape[0]

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1,2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1,2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1,2)

        scores = torch.matmul(Q, K.transpose(-2,-1))

        scores = scores / (self.head_dim ** 0.5)

        attention = F.softmax(scores, dim=-1)

        out = torch.matmul(attention, V)

        out = out.transpose(1,2).contiguous()

        out = out.view(batch_size, -1, self.embedding_dim)

        out = self.fc_out(out)

        return out, attention

In [6]:
multi_head = MultiHeadSelfAttention(
    embedding_dim=8,
    num_heads=2
)

output, attention = multi_head(x)

print("Output Shape:", output.shape)
print(output)

print("\nAttention Shape:", attention.shape)
print(attention)

Output Shape: torch.Size([2, 5, 8])
tensor([[[ 0.2133,  0.2117, -0.0541, -0.3263,  0.2037, -0.2190,  0.1064,
           0.4124],
         [ 0.1853,  0.2314,  0.0066, -0.4176,  0.2550, -0.1046,  0.1304,
           0.3671],
         [ 0.2057,  0.3006, -0.0168, -0.3310,  0.2774, -0.0920, -0.0033,
           0.3509],
         [ 0.1373,  0.2986,  0.0055, -0.3642,  0.2686, -0.0976,  0.0709,
           0.3225],
         [ 0.0830,  0.2161, -0.0231, -0.4205,  0.2335, -0.1890,  0.2530,
           0.3083]],

        [[ 0.4339,  0.0620, -0.2467, -0.4989,  0.1072, -0.5010,  0.6770,
           0.4730],
         [ 0.4033,  0.1025, -0.2407, -0.4681,  0.1012, -0.4749,  0.5988,
           0.4825],
         [ 0.4520,  0.0717, -0.2384, -0.5038,  0.1130, -0.4822,  0.6449,
           0.4873],
         [ 0.3982,  0.1195, -0.2418, -0.4460,  0.1032, -0.4642,  0.5636,
           0.4896],
         [ 0.3438,  0.1051, -0.2513, -0.4234,  0.0751, -0.4877,  0.5895,
           0.4877]]], grad_fn=<ViewBackward0>)

Atte

In [7]:
class MaskedSelfAttention(nn.Module):

    def __init__(self, embedding_dim):

        super(MaskedSelfAttention, self).__init__()

        self.query = nn.Linear(embedding_dim, embedding_dim)
        self.key = nn.Linear(embedding_dim, embedding_dim)
        self.value = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = torch.matmul(Q, K.transpose(-2,-1))

        scores = scores / (K.size(-1) ** 0.5)

        seq_length = x.size(1)

        mask = torch.tril(torch.ones(seq_length, seq_length))

        scores = scores.masked_fill(mask == 0, float('-inf'))

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        return output, attention

In [8]:
masked_attention = MaskedSelfAttention(embedding_dim)

output, attention = masked_attention(x)

print("Output Shape:", output.shape)
print(output)

print("\nMasked Attention Weights:")
print(attention)

Output Shape: torch.Size([2, 5, 8])
tensor([[[-7.6358e-01, -2.1479e+00, -3.5714e-01, -6.2899e-01, -2.1772e-01,
          -7.7883e-04, -5.4760e-01, -1.4649e+00],
         [ 4.4701e-02, -1.5362e+00,  4.9206e-02, -1.1218e-01, -1.1137e-01,
           3.7009e-01, -1.8070e-01, -6.7442e-01],
         [ 2.8943e-01, -8.1816e-01,  2.1750e-01, -1.5546e-01, -3.0354e-01,
          -9.3358e-02, -3.4413e-01,  7.6145e-02],
         [ 7.1206e-02, -6.0234e-01,  7.3344e-02, -1.8927e-01, -5.5276e-01,
           2.0756e-03, -3.9735e-01,  1.5668e-01],
         [-1.4918e-01, -3.1115e-01, -3.7987e-02, -5.2364e-01, -3.8559e-01,
           1.5769e-01, -2.4308e-01,  1.0797e-01]],

        [[ 9.5323e-01, -9.4228e-01,  9.5417e-01,  2.3357e-01,  1.1282e+00,
           4.7717e-01, -3.7454e-01,  1.1061e+00],
         [ 6.7321e-01, -4.4138e-01,  7.8334e-01,  2.3169e-02,  7.4158e-01,
           2.0626e-01, -4.0158e-01,  1.1920e+00],
         [ 8.2160e-01, -1.4654e-01,  8.4385e-01,  3.1464e-01,  4.6375e-01,
           2

In [9]:
seq_length = 5

mask = torch.tril(torch.ones(seq_length, seq_length))

print(mask)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])
